Inicialização do ambiente

In [1]:
#Import bibliotecas
import os
import sys
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, current_timestamp, from_json, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [2]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [3]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [4]:
#Iniciando sessão spark
spark = SparkSession.builder \
    .appName("bronze-layer") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e5b2f45e-b0b4-4573-bd17-7307814e57a0;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.6.0/postgresql-42.6.0.jar ...
	[SUCCESSFUL ] org.postgresql#postgresql;42.6.0!postgresql.jar (163ms)
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.4.2/iceberg-spark-runtime-3.5_2.12-1.4.2.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2!iceberg-spark-runtime-3.5_2.12.jar (506ms)
downloading https://repo1.ma

In [5]:
!ls /tmp/warehouse/bronze/transacoes

ls: cannot access '/tmp/warehouse/bronze/transacoes': No such file or directory


In [6]:
#Variaveis de conexao ao ambiente postgree
conexao = {
           "url": "jdbc:postgresql://postgres:5432/pipeline_transacoes",
           "user": "postgres",
           "password": "1234",
           "driver": "org.postgresql.Driver"
}

Processamento de dados

In [7]:
#Dados vindo da API
df_raw = spark.read.jdbc(
    url=conexao["url"],
    table="staging.transacoes_raw",  # tabela no PostgreSQL
    properties={
        "user": conexao["user"],
        "password": conexao["password"],
        "driver": conexao["driver"]
    }
)

df_raw.select("payload").show(truncate=True)

+--------------------+
|             payload|
+--------------------+
|{"valor": 4029.14...|
|{"valor": 3871.92...|
|{"valor": 1601.04...|
|{"valor": null, "...|
|{"valor": 4593.94...|
|{"valor": 3753.94...|
|{"valor": 835.53,...|
|{"valor": 3748.62...|
|{"valor": 3454.13...|
|{"valor": 1013.46...|
|{"valor": 2466.9,...|
|{"valor": 3533.54...|
|{"valor": 3770.25...|
|{"valor": 615.21,...|
|{"valor": 1485.56...|
|{"valor": 2624.44...|
|{"valor": 4852.29...|
|{"valor": 1363.97...|
|{"valor": 1819.36...|
|{"valor": 2802.25...|
+--------------------+
only showing top 20 rows



In [14]:
#Gravando no formato parquet
df_bronze.write.partitionBy("data_particao").mode("overwrite").parquet("/tmp/parquet_puro")

# Ler o Parquet gravado
df_parquet = spark.read.parquet("/tmp/parquet_puro")

# Medir tempo da query filtrando
start = time.time()
df_parquet_filtered = df_parquet.filter("data_particao = '2026-04-12'")
df_parquet_filtered.show()
print(f"Parquet puro - Tempo de execução: {time.time() - start} segundos")

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  1|4029.14|notebook|eletronicos|        11|         2|  cartao_credito|2026-04-12 01:42:...|   2026-04-12|
|  2|3871.92|    fone|eletronicos|        28|         2|             pix|2026-04-12 01:42:...|   2026-04-12|
|  3|1601.04|notebook|eletronicos|        18|         2|          boleto|2026-04-12 01:42:...|   2026-04-12|
|  4|   NULL| celular|eletronicos|        92|         2|          boleto|2026-04-12 01:42:...|   2026-04-12|
|  5|4593.94|   tenis|  vestuario|        25|         2|          boleto|2026-04-12 01:42:...|   2026-04-12|
|  6|3753.94|camiseta|  vestuario|        57|         3|  cartao_credito|2026-04-12 01:42:...|   2026-04-12|
|  7| 835.53|camise

In [8]:
#Criando tabela bronze se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

#Definir schema do payload
schema_payload = StructType([
    StructField("valor", DoubleType(), True),
    StructField("produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("quantidade", IntegerType(), True),
    StructField("metodo_pagamento", StringType(), True),
])

# Transformar DF raw
df_bronze = (
    df_raw
    .withColumn("data_ingestao", current_timestamp())
    .withColumn("data_particao", to_date(col("data_ingestao")))
    .withColumn("payload_struct", from_json(col("payload"), schema_payload))
    .select(
        "id",
        "payload_struct.*",
        "data_ingestao",
        "data_particao"
    )
)                           
df_bronze.writeTo("lakehouse.bronze.transacoes") \
         .using("iceberg") \
         .partitionedBy("data_particao") \
         .createOrReplace()

26/04/12 01:41:20 WARN HadoopTableOperations: Error reading version hint file /tmp/warehouse/bronze/transacoes/metadata/version-hint.text
java.io.FileNotFoundException: File /tmp/warehouse/bronze/transacoes/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apache.iceberg.hadoop.HadoopTableOper

In [12]:
start = time.time()
df_iceberg_filtered = spark.read.format("iceberg").load("lakehouse.bronze.transacoes").filter("data_particao='2026-04-12'")
df_iceberg_filtered.show()
print(f"Iceberg - Tempo de execução: {time.time() - start} segundos")

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  1|4029.14|notebook|eletronicos|        11|         2|  cartao_credito|2026-04-12 01:41:...|   2026-04-12|
|  2|3871.92|    fone|eletronicos|        28|         2|             pix|2026-04-12 01:41:...|   2026-04-12|
|  3|1601.04|notebook|eletronicos|        18|         2|          boleto|2026-04-12 01:41:...|   2026-04-12|
|  4|   NULL| celular|eletronicos|        92|         2|          boleto|2026-04-12 01:41:...|   2026-04-12|
|  5|4593.94|   tenis|  vestuario|        25|         2|          boleto|2026-04-12 01:41:...|   2026-04-12|
|  6|3753.94|camiseta|  vestuario|        57|         3|  cartao_credito|2026-04-12 01:41:...|   2026-04-12|
|  7| 835.53|camise

In [10]:
#Select tabela transacoes na bronze
spark.sql("SELECT * FROM lakehouse.bronze.transacoes LIMIT 5").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  1|4029.14|notebook|eletronicos|        11|         2|  cartao_credito|2026-04-12 01:41:...|   2026-04-12|
|  2|3871.92|    fone|eletronicos|        28|         2|             pix|2026-04-12 01:41:...|   2026-04-12|
|  3|1601.04|notebook|eletronicos|        18|         2|          boleto|2026-04-12 01:41:...|   2026-04-12|
|  4|   NULL| celular|eletronicos|        92|         2|          boleto|2026-04-12 01:41:...|   2026-04-12|
|  5|4593.94|   tenis|  vestuario|        25|         2|          boleto|2026-04-12 01:41:...|   2026-04-12|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+



In [ ]:
#Validando estrutura tabela bronze
spark.sql("Describe lakehouse.bronze.transacoes").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|                  id|      int|   NULL|
|               valor|   double|   NULL|
|             produto|   string|   NULL|
|           categoria|   string|   NULL|
|          cliente_id|      int|   NULL|
|          quantidade|      int|   NULL|
|    metodo_pagamento|   string|   NULL|
|       data_ingestao|timestamp|   NULL|
|       data_particao|     date|   NULL|
|# Partition Infor...|         |       |
|          # col_name|data_type|comment|
|       data_particao|     date|   NULL|
+--------------------+---------+-------+



Validação dos dados (Bronze) para posteriormente realizar as transformações necessarias (Silver)

In [ ]:
#Validando a existencia de dados nulos na bronze
from pyspark.sql.functions import col, sum
df_bronze.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_bronze.columns
]).show()

+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
| id|valor|produto|categoria|cliente_id|quantidade|metodo_pagamento|data_ingestao|data_particao|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+
|  0|   11|      0|        0|         0|         0|               0|            0|            0|
+---+-----+-------+---------+----------+----------+----------------+-------------+-------------+



In [ ]:
#Verificando dados nulos no campo VALOR
spark.sql("SELECT * FROM lakehouse.bronze.transacoes where valor is null").show()

+---+-----+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-----+--------+-----------+----------+----------+----------------+--------------------+-------------+
| 13| NULL|   tenis|  vestuario|        71|         1|             pix|2026-03-30 01:55:...|   2026-03-30|
| 14| NULL|   tenis|  vestuario|        47|         1|          boleto|2026-03-30 01:55:...|   2026-03-30|
| 27| NULL|    fone|eletronicos|         3|         2|             pix|2026-03-30 01:55:...|   2026-03-30|
| 30| NULL| celular|eletronicos|        43|         1|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
| 33| NULL|notebook|eletronicos|        35|         2|             pix|2026-03-30 01:55:...|   2026-03-30|
| 34| NULL| celular|eletronicos|        48|         2|          boleto|2026-03-30 01:55:...|   2026-03-30|
| 40| NULL|camiseta|  vestuario|     

In [ ]:
#Validando se a duplicata para possivel tratamento
df_bronze.groupBy("id", "data_ingestao") \
         .agg(count("*").alias("qtd")) \
         .filter(col("qtd") > 1) \
         .show()

+---+-------------+---+
| id|data_ingestao|qtd|
+---+-------------+---+
+---+-------------+---+



Transformação de Dados (Silver)

In [ ]:
#Gerando tabela transacao SILVER com base na bronze porem com tratamento de campos null

# Ler Bronze
df_bronze = spark.read.table("lakehouse.bronze.transacoes")

#Tratamento valor null por 0
df_curated = (
    df_bronze.fillna({"valor": 0})
)

# Criar database Curated se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.silver")

# Salvar Curated em Delta Lake, particionado por data
df_curated.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()


spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+
|  1|3716.46|camiseta|  vestuario|        11|         1|             pix|2026-03-30 01:55:...|   2026-03-30|
|  2|2860.04|   tenis|  vestuario|         1|         1|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
|  3|4207.49| celular|eletronicos|        80|         2|          boleto|2026-03-30 01:55:...|   2026-03-30|
|  4| 429.82|   tenis|  vestuario|        43|         3|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
|  5| 787.63|    fone|eletronicos|        19|         1|             pix|2026-03-30 01:55:...|   2026-03-30|
|  6|1772.99|notebook|eletronicos|        95|         3|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|
|  7|  70.52|camise

In [ ]:
#Atualização na tabela silver (ADD COLUMNS)
df_add = df_curated.withColumn("preco_unitario", (col("valor") / col("quantidade")))
df_add.writeTo("lakehouse.silver.transacoes") \
          .partitionedBy("data_particao") \
          .createOrReplace()

In [ ]:
#Validando após alteração
spark.sql("SELECT * FROM lakehouse.silver.transacoes").show()

+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
| id|  valor| produto|  categoria|cliente_id|quantidade|metodo_pagamento|       data_ingestao|data_particao|    preco_unitario|
+---+-------+--------+-----------+----------+----------+----------------+--------------------+-------------+------------------+
|  1|3716.46|camiseta|  vestuario|        11|         1|             pix|2026-03-30 01:55:...|   2026-03-30|           3716.46|
|  2|2860.04|   tenis|  vestuario|         1|         1|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|           2860.04|
|  3|4207.49| celular|eletronicos|        80|         2|          boleto|2026-03-30 01:55:...|   2026-03-30|          2103.745|
|  4| 429.82|   tenis|  vestuario|        43|         3|  cartao_credito|2026-03-30 01:55:...|   2026-03-30|143.27333333333334|
|  5| 787.63|    fone|eletronicos|        19|         1|             pix|2026-03-30 01:55:...|   2026-03

Analise de perfomance (snapshot)

In [ ]:
spark.sql("""SELECT 
  snapshot_id,
  committed_at,
  operation,
  summary['operation-type'],
  summary['added-data-files'],
  summary['removed-data-files']
FROM lakehouse.silver.transacoes.snapshots
ORDER BY committed_at DESC""").show()

Com base na analise de snapshot da tabela silver podemos notar que só foi feito append, ou seja, só houve adição de dados
nenhum arquivo foi removido ou reescrito. O append é uma operação de baixo custo, pois não há custo de reescrita.
No nosso cenario é gerado novos dados já com a coluna calculada e inseridos na tabela, gerandoa alerta de perfomance somente relacionado a small files.